# Week 23, Day 5: Dates, times and time zones

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w023d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from datetime import datetime, timezone, timedelta

lagos = timezone(timedelta(hours=1))
meeting = datetime(2026, 11, 2, 15, 0, tzinfo=lagos)
print(meeting)
print(meeting.astimezone(timezone.utc))

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

lagos = ZoneInfo("Africa/Lagos")
meeting = datetime(2026, 11, 2, 15, 0, tzinfo=lagos)
print(meeting, meeting.tzname(), meeting.utcoffset())

now_utc = datetime(2026, 11, 2, 14, 0, tzinfo=ZoneInfo("UTC"))
print(meeting == now_utc)          # the same moment

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

call = datetime(2026, 11, 2, 15, 0, tzinfo=ZoneInfo("Africa/Lagos"))
for place in ["UTC", "Europe/London", "Africa/Nairobi", "America/New_York", "Asia/Kolkata", "Australia/Sydney"]:
    local = call.astimezone(ZoneInfo(place))
    print(f"{place:<18} {local:%a %d %b %H:%M} ({local.tzname()})")

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

london = ZoneInfo("Europe/London")
first = datetime(2026, 10, 25, 1, 30, tzinfo=london)            # fold=0: the first 01:30 (BST)
second = datetime(2026, 10, 25, 1, 30, fold=1, tzinfo=london)   # the second 01:30 (GMT)
print(first.utcoffset(), second.utcoffset())

start = datetime(2026, 10, 25, 0, 0, tzinfo=london)
end = datetime(2026, 10, 26, 0, 0, tzinfo=london)
utc = ZoneInfo("UTC")
print(end.astimezone(utc) - start.astimezone(utc))      # how long that day really was

> **Key idea.** **Store and calculate in UTC; convert to local time only to show it to a person.** Make datetimes aware with `ZoneInfo("Area/City")`, never mix naive and aware, use `astimezone` to convert, and do arithmetic across daylight saving changes in UTC.

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

when = datetime(2026, 11, 2, 15, 0, tzinfo=ZoneInfo("Africa/Lagos"))
stored = when.astimezone(timezone.utc).isoformat()
print(stored)
back = datetime.fromisoformat(stored)
print(back.astimezone(ZoneInfo("Africa/Lagos")))

ts = 1_800_000_000
print(datetime.fromtimestamp(ts, timezone.utc), int(back.timestamp()))

### Exercise: A world clock

Write `world_clock(moment, places)` that takes an aware datetime and a list of IANA zone names, and returns a list of strings like `"Lagos 15:00 Mon"` for each place: the part of the name after the last `/` (with `_` turned into a space), then the local time `HH:MM` and the short weekday.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    from datetime import datetime\n    from zoneinfo import ZoneInfo\n    m = datetime(2026, 11, 2, 14, 0, tzinfo=ZoneInfo(\"UTC\"))\n    assert world_clock(m, [\"Africa/Lagos\", \"America/New_York\", \"Australia/Sydney\", \"Asia/Kolkata\"]) == [\n        \"Lagos 15:00 Mon\", \"New York 09:00 Mon\", \"Sydney 01:00 Tue\", \"Kolkata 19:30 Mon\"]")

### Exercise: When is the meeting for me?

Write `local_time(text, from_zone, to_zone)` where `text` is a naive time written `YYYY-MM-DD HH:MM` that's local to `from_zone`; return the same moment as `YYYY-MM-DD HH:MM` in `to_zone`. Attach the first zone with `replace`, then convert with `astimezone`.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert local_time(\"2026-11-02 15:00\", \"Africa/Lagos\", \"Europe/London\") == \"2026-11-02 14:00\"\n    assert local_time(\"2026-07-02 15:00\", \"Africa/Lagos\", \"Europe/London\") == \"2026-07-02 15:00\"\n    assert local_time(\"2026-11-02 23:30\", \"Europe/London\", \"Asia/Tokyo\") == \"2026-11-03 08:30\"\n    assert local_time(\"2026-11-02 09:00\", \"America/New_York\", \"Africa/Lagos\") == \"2026-11-02 15:00\"")

### Exercise: How far apart?

Write `offset_hours(zone, when)` returning a zone's UTC offset in hours (a float) at the naive datetime `when` (interpreted in that zone), and `difference(zone_a, zone_b, when)` returning how many hours `zone_a` is ahead of `zone_b` at that moment. London and Lagos are 0 hours apart in summer and 1 apart in winter.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    from datetime import datetime\n    assert offset_hours(\"Africa/Lagos\", datetime(2026, 1, 1)) == 1.0 and offset_hours(\"Asia/Kolkata\", datetime(2026, 1, 1)) == 5.5\n    assert offset_hours(\"Europe/London\", datetime(2026, 7, 1)) == 1.0 and offset_hours(\"Europe/London\", datetime(2026, 1, 1)) == 0.0\n    assert difference(\"Africa/Lagos\", \"Europe/London\", datetime(2026, 1, 15)) == 1.0\n    assert difference(\"Africa/Lagos\", \"Europe/London\", datetime(2026, 7, 15)) == 0.0\n    assert difference(\"America/New_York\", \"Africa/Lagos\", datetime(2026, 1, 15)) == -6.0")

### Exercise: The long day and the short day

Write `hours_in_day(year, month, day, zone)` returning the real number of hours from midnight to the next midnight in that zone (convert both to UTC before subtracting), and `dst_changes(year, zone)` returning a list of the dates (as `date` objects) in that year whose length isn't 24 hours.

In [ ]:
from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    from datetime import date\n    assert hours_in_day(2026, 10, 25, \"Europe/London\") == 25\n    assert hours_in_day(2026, 3, 29, \"Europe/London\") == 23\n    assert hours_in_day(2026, 6, 1, \"Europe/London\") == 24 and hours_in_day(2026, 10, 25, \"Africa/Lagos\") == 24\n    assert dst_changes(2026, \"Europe/London\") == [date(2026, 3, 29), date(2026, 10, 25)]\n    assert dst_changes(2026, \"Africa/Lagos\") == []")

### Exercise: Timestamps from other systems

Three systems log the same events differently. Write `to_utc(stamp)` that turns any of these into an aware UTC datetime: an ISO string with an offset (`"2026-11-02T15:00:00+01:00"`), an ISO string ending in `Z` (`"2026-11-02T14:00:00Z"`), or a Unix timestamp given as an int or a string of digits (`1793628000`). Then `sort_events(stamps)` returning the stamps (as given) in time order.

In [ ]:
from datetime import datetime, timezone

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    from datetime import datetime, timezone\n    want = datetime(2026, 11, 2, 14, 0, tzinfo=timezone.utc)\n    assert to_utc(\"2026-11-02T15:00:00+01:00\") == want and to_utc(\"2026-11-02T14:00:00Z\") == want\n    assert to_utc(int(want.timestamp())) == want and to_utc(str(int(want.timestamp()))) == want\n    assert to_utc(\"2026-11-02T15:00:00+01:00\").utcoffset().total_seconds() == 0\n    early = int(want.timestamp()) - 7200\n    stamps = [\"2026-11-02T15:30:00+01:00\", early, \"2026-11-02T13:00:00Z\"]\n    assert sort_events(stamps) == [early, \"2026-11-02T13:00:00Z\", \"2026-11-02T15:30:00+01:00\"]")

### Bug hunt: Naive meets aware

This crashes with `TypeError: can't compare offset-naive and offset-aware datetimes`, because the deadline is naive. The deadline is 17:00 London time. Make it aware, so the program prints `late`.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

deadline = datetime(2026, 11, 2, 17, 0)
submitted = datetime(2026, 11, 2, 18, 30, tzinfo=ZoneInfo("Africa/Lagos"))
print("late" if submitted > deadline else "on time")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"late\")")

### Bug hunt: replace is not astimezone

This should show a 15:00 Lagos meeting in New York time, `10:00`, but prints `15:00`: `replace` just relabels the clock reading with a new zone, which changes the moment. Fix it so it converts.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

meeting = datetime(2026, 11, 2, 15, 0, tzinfo=ZoneInfo("Africa/Lagos"))
ny = meeting.replace(tzinfo=ZoneInfo("America/New_York"))
print(ny.strftime("%H:%M"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"09:00\")")

### Puzzle: Store in UTC, show locally

Arrange the lines so the program stores a local event time as a UTC ISO string, then reads it back and shows it in Nairobi time. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # the offset from UTC of Lagos, in hours
b = ___   # True or False: datetime(2026, 1, 1) is aware
c = ___   # the module in the standard library with real time zones by name
d = ___   # how many hours long is the day the clocks go back in London?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 1 and b is False and c == \"zoneinfo\" and d == 25")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: find a meeting time

A team is spread across time zones, and each person is happy to meet between 09:00 and 17:00 their local time on weekdays. Write `overlap(day, zones)` returning a list of the whole UTC hours (0 to 23) on the given UTC date (a `date`) when **every** person is within their working hours (a meeting starting at that hour must also end by 17:00, so the hour counts if local start is from 09:00 up to 16:00) and it's a weekday (Monday to Friday) where they are. Then `best_slot(day, zones, show_in)` returning the earliest overlapping hour as a string like `"14:00 UTC = 15:00 in Africa/Lagos"` (shown in the `show_in` zone), or `"no overlap"`.

In [ ]:
from datetime import datetime, date, timezone
from zoneinfo import ZoneInfo

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    from datetime import date\n    team = [\"Africa/Lagos\", \"Europe/London\", \"America/New_York\"]\n    assert overlap(date(2026, 11, 2), team) == [14, 15]\n    assert best_slot(date(2026, 11, 2), team, \"Africa/Lagos\") == \"14:00 UTC = 15:00 in Africa/Lagos\"\n    assert overlap(date(2026, 7, 1), team) == [13, 14, 15]\n    assert overlap(date(2026, 11, 2), [\"Africa/Lagos\", \"Australia/Sydney\"]) == []\n    assert best_slot(date(2026, 11, 2), [\"Africa/Lagos\", \"Australia/Sydney\"], \"UTC\") == \"no overlap\"\n    assert overlap(date(2026, 11, 1), [\"Europe/London\"]) == []")

## Warm-up

3pm in Lagos and 2pm in UTC are the **same moment**, written for two places. The `+01:00` on the end says how far that clock is from UTC. Once a datetime carries that information, Python can convert it to any other place's clock, and compare it properly with any other time.

## UTC and time zones

**UTC** (Coordinated Universal Time) is the world's reference clock: no time zone, no daylight saving, never jumps. Every place's local time is UTC plus or minus an **offset**:

| Place | Usual offset | Notes |
|---|---|---|
| Lagos, Kinshasa | UTC+1 | West Africa Time, no daylight saving |
| London | UTC+0 in winter, UTC+1 in summer | British Summer Time from late March to late October |
| Nairobi | UTC+3 | East Africa Time |
| New York | UTC−5 in winter, UTC−4 in summer | different change dates from Europe |
| Kolkata | UTC+5:30 | offsets aren't always whole hours |

A **time zone** is more than an offset: it's a region's **rules**, including when its daylight saving starts and ends, and their history (countries change them). Those rules are kept in the **IANA time zone database**, with names like `Africa/Lagos`, `Europe/London` and `America/New_York`. Always use these names, never abbreviations like "BST" or "EST", which are ambiguous.

## Naive and aware

A Python `datetime` is either:

- **naive**: no time zone attached (`tzinfo` is `None`). `datetime(2026, 11, 2, 15, 0)` is "3pm somewhere". Everything in Week 20 was naive.
- **aware**: with a time zone attached. It names an exact moment, which can be compared with any other aware time and converted to any zone.

The `zoneinfo` module (in the standard library since Python 3.9) gives you real time zones by name:

(On your own computer this works straight away on macOS and Linux. On Windows, `pip install tzdata` first, since Windows doesn't include the IANA database. This site loads it for you.)

Two rules prevent most problems:

- **Never mix naive and aware.** Comparing or subtracting them raises `TypeError: can't compare offset-naive and offset-aware datetimes`, and that error is a gift: it catches a bug.
- **Get the current time as aware:** `datetime.now(timezone.utc)` or `datetime.now(ZoneInfo("Africa/Lagos"))`, never plain `datetime.now()` for anything stored or compared. (`datetime.utcnow()` is deprecated: it returns a naive time that only looks like UTC.)

## Converting between zones

`astimezone(zone)` converts an aware datetime to another zone's clock. The moment doesn't change; only how it's written:

Note the day: 15:00 in Lagos is already Tuesday morning in Sydney. Converting times always means converting dates too.

To **attach** a zone to a naive time you know is local (from a form or a file), use `replace(tzinfo=...)`; to **convert**, use `astimezone`. Mixing those up is a classic bug: `replace` keeps the clock reading and changes the moment; `astimezone` keeps the moment and changes the clock reading.

## Daylight saving time

Places with daylight saving (DST) move their clocks forward one hour in spring and back one hour in autumn. That has odd consequences:

- In spring, an hour **doesn't exist**: in London on 29 March 2026, the clocks go from 00:59:59 straight to 02:00. "01:30 that day" isn't a real time there.
- In autumn, an hour **happens twice**: on 25 October 2026, London's clocks show 01:00 to 01:59 once in BST (UTC+1), then again in GMT (UTC+0).
- A "day" isn't always 24 hours long: that autumn Sunday in London lasts 25 hours.

`zoneinfo` handles all of this, and the `fold` attribute (0 or 1) chooses between the first and second time a repeated hour happens:

**Arithmetic on aware datetimes in the same zone uses clock time**, not elapsed time: adding `timedelta(hours=24)` to midnight on the long day gives midnight the next day, even though 25 real hours passed. To work with **real elapsed time**, convert to UTC, do the arithmetic, and convert back. This is the main reason for the rule below.

## Storing, parsing and measuring

**Storing and exchanging:** write times in UTC, in **ISO 8601** form: `2026-11-02T14:00:00+00:00` (or with `Z` for UTC, `2026-11-02T14:00:00Z`). `isoformat()` writes it and `datetime.fromisoformat()` reads it back, offset included (it accepts `Z` from Python 3.11). Databases, JSON APIs and logs almost all use this.

**Unix timestamps:** many systems record time as seconds since 1 January 1970 UTC (the "Unix epoch"). Convert with `datetime.fromtimestamp(ts, timezone.utc)` and `dt.timestamp()`. Always pass `timezone.utc`, or you get a naive local time.

**Measuring how long something takes:** don't subtract two `datetime.now()` readings, because the computer's clock can be adjusted (by the network, or by you) while code runs. Use `time.monotonic()` (or `time.perf_counter()`, from Week 17), which only ever goes forwards:

```
import time
start = time.monotonic()
do_work()
elapsed = time.monotonic() - start
```

## Practice

## Quiz

## Daily challenge

## Before you go

Check the time difference between where you are and two places you know someone, in both January and July, with `world_clock`. Notice whether either place has daylight saving. Tomorrow is the Forge: a data-cleaning tool that brings regex, JSON, CSV and time zones together.